# Country similarity pool ranking based on SINAS dataset

This notebook explores how to use the SINAS dataset (Goméz-Suárez et al., 2025) to create a horizon scanning priority species list. For it to run, please download the latest version of  the SINAS dataset from [this link](https://doi.org/10.5281/zenodo.18220953) and save it in the 'data' directory of the repository.

## 1. Setup

In [1]:
#import dependencies

import pandas as pd
from pathlib import Path
import sys
import os

print("All dependencies imported successfully.")

All dependencies imported successfully.


## 2. Load in the SINAS dataset and preview file

In [2]:
#set file path
repo_root = Path.cwd().parent
file_path = repo_root / 'data' / 'SInAS_3.1.1.csv' #adapt this to local file name

print({file_path}) #verify if correct location

raw_data = raw_data = pd.read_csv(
    file_path, 
    sep=None, 
    engine='python', #autodetect
    quotechar='"' #seperator
)

print("Data loaded successfully.")
print(f"Columns found: {raw_data.columns.tolist()}") #show column types

raw_data.head() #show df preview



{WindowsPath('c:/Users/simon/Documents/GitHub/horizon-scanner/data/SInAS_3.1.1.csv')}
Data loaded successfully.
Columns found: ['location', 'locationID', 'taxon', 'taxonID', 'eventDate', 'habitat', 'occurrenceStatus', 'establishmentMeans', 'degreeOfEstablishment', 'pathway', 'datasetName', 'bibliographicCitation']


,location,locationID,taxon,taxonID,eventDate,habitat,occurrenceStatus,establishmentMeans,degreeOfEstablishment,pathway,datasetName,bibliographicCitation
0,Aegean,101,Aphaenogaster splendida,5461,NaN,NaN,NaN,introduced,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su..."
1,Aegean,101,Camponotus fallax,5586,NaN,terrestrial,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su..."
2,Aegean,101,Camponotus vagus,5601,NaN,terrestrial,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su..."
3,Aegean,101,Cardiocondyla mauritanica,5609,NaN,terrestrial,NaN,introduced,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su..."
4,Aegean,101,Cataglyphis nodus,5620,NaN,NaN,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su..."


## 3. Verify if EU member states and surrounding areas are in location column 

In [3]:
# 1. Define the target countries
eu_27 = [
    'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic', 
    'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 
    'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 
    'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 
    'Slovenia', 'Spain', 'Sweden'
]

uk = ['United Kingdom'] # You may need to add 'UK' or 'Great Britain' if your data uses those

# Broad list of European neighbors (EFTA, Balkans, Eastern Europe)
neighbors = [
    'Albania', 'Andorra', 'Belarus', 'Bosnia And Herzegovina', 'Iceland', 
    'Kosovo', 'Liechtenstein', 'Moldova', 'Monaco', 'Montenegro', 
    'North Macedonia', 'Norway', 'Russia', 'San Marino', 'Serbia', 
    'Switzerland', 'Turkey', 'Ukraine', 'Vatican City'
]

# Combine all lists into a single set for easy mathematical comparison
target_countries = set(eu_27 + uk + neighbors)

# 2. Extract unique locations from your dataset
# We drop NAs and convert to title case just in case there are lowercase entries
available_locations = set(
    raw_data['location']
    .dropna()
    .astype(str)
    .str.title()
    .unique()
    )

# 3. Calculate present and missing countries
present_countries = target_countries.intersection(available_locations)
missing_countries = target_countries.difference(available_locations)

# 4. Output the results
print("--- LOCATION CHECK RESULTS ---")
print(f"Total target countries: {len(target_countries)}")
print(f"Countries found in data: {len(present_countries)}")
print(f"Countries missing: {len(missing_countries)}\n")

if missing_countries:
    print("Missing Countries (Check for spelling/naming variations like 'Czechia' vs 'Czech Republic'):")
    for country in sorted(missing_countries):
        print(f" - {country}")
else:
    print("✅ All EU member states, the UK, and neighboring countries are present in the 'location' column!")

--- LOCATION CHECK RESULTS ---
Total target countries: 47
Countries found in data: 47
Countries missing: 0

✅ All EU member states, the UK, and neighboring countries are present in the 'location' column!


## 4. Get similarity in presence / absence between regions

In [5]:
sys.path.append(os.path.abspath('..')) #make sure our base path is set correctly

#import dependencies

from src.get_region_ranking import rank_regions_by_similarity #import function

#specify the target country

target_region = "Belgium"

#Query for rankings and similarity scores, comparing all species (not just introduced ones)

try:
    # Get the rankings (comparing all species)
    rankings = rank_regions_by_similarity(raw_data, target_region, compare_only_introduced=False)
    
    print(f"--- Top 100 Regions Most Similar to {target_region} ---")
    
    # Format the similarity score for a cleaner display
    display_df = rankings.head(100).copy()
    display_df['Similarity score'] = (display_df['Similarity score'] * 100).round(2).astype(str) + '%'
    
    display(display_df) # Use display() instead of print() in notebooks for a nicely formatted table

except ValueError as e:
    print(f"Error: {e}")

--- Top 100 Regions Most Similar to Belgium ---


,Region,Shared species,Total unique species in both,Target region total,Comparison region total,Similarity score
0,Netherlands,3274,6280,5007,4547,52.13%
1,United Kingdom,3125,7323,5007,5441,42.67%
2,Austria,2710,6589,5007,4292,41.13%
3,Switzerland,2415,5872,5007,3280,41.13%
4,Denmark,2575,6391,5007,3959,40.29%
...,...,...,...,...,...,...
95,Yemen,384,5874,5007,1251,6.54%
96,Israel,375,5952,5007,1320,6.3%
97,Peru,417,6805,5007,2215,6.13%
98,Tanzania,413,6907,5007,2313,5.98%


## 5. Get IAS risk probability score based on similarity with other regions

In [6]:
sys.path.append(os.path.abspath('..')) #make sure our base path is set correctly

#import dependencies

from src.get_region_ranking import predict_ias_risk #import function

# set target region for risk prediction

target_region_name = "Belgium"
horizon_scan_list_length = 100

# 1. First, get the similarity scores for the target region. 
# (We compare against all species to get a true ecological baseline)
similarity_rankings = rank_regions_by_similarity(
    raw_data, 
    target_region_name, 
    compare_only_introduced=False
    )

# 2. Pass that similarity dataframe into the risk predictor
ias_risk_predictions = predict_ias_risk(
    df=raw_data, 
    similarity_df=similarity_rankings, 
    target_region=target_region_name,
    species_to_validate=horizon_scan_list_length 
)

print(f"--- Top IAS Threats for {target_region_name} ---")

# Display the top 10 highest-risk species
display_df = ias_risk_predictions.copy()

# Format the floats for cleaner reading
display_df['Cumulative risk score'] = display_df['Cumulative risk score'].round(3)
display_df['Max single-region similarity'] = (display_df['Max single-region similarity'] * 100).round(2).astype(str) + '%'

display(display_df)

Ranking species locally first...
Validating top threats against GBIF for 'BE'...


True threats found:   0%|          | 0/100 [00:00<?, ?species/s]

Done! Evaluated 352 species via GBIF to isolate 100 validated threats.
--- Top IAS Threats for Belgium ---


,Species,Cumulative risk score,Max single-region similarity,Found in regions,Region count
0,Cuscuta pentagona,12.931,52.13%,"Albania, Algeria, Argentina, Armenia, Banglade...",90
1,Plum pox virus,12.249,52.13%,"Albania, Argentina, Austria, Azores, Belarus, ...",53
2,Gambusia affinis,10.606,39.68%,"Afghanistan, Albania, Algeria, American Samoa,...",88
3,Chenopodium ambrosioides,9.156,52.13%,"Albania, Angola, Anguilla, Australia, Austria,...",51
4,Pheidole megacephala,8.749,42.67%,"Anguilla, Antigua And Barbuda, Argentina, Arub...",127
...,...,...,...,...,...
95,Aphis spiraephaga,4.090,41.13%,"Austria, Bulgaria, Croatia, Czech Republic, De...",14
96,Illinoia azaleae,4.085,42.67%,"Austria, Czech Republic, France, Germany, Hawa...",13
97,Gnatocerus cornutus,4.084,52.13%,"Austria, Estonia, France, Galapagos, Germany, ...",14
98,Rubus allegheniensis,4.084,52.13%,"Canada, Czech Republic, Denmark, Germany, Japa...",12
